# 🗺️ Project 9.4 — Weighted Robot Path Planner
**DSA for Mechatronics · Week 09 — Graphs**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq
from collections import defaultdict, deque, Counter
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A mobile robot navigates a factory floor modelled as a **weighted directed graph**
where edge weights represent travel time in seconds.

We apply three algorithms:
1. **Dijkstra's shortest path** — find the fastest route between two points
2. **Multi-source BFS** — simultaneously spread from multiple charging stations,
   find the nearest charger for every node
3. **Grid flood fill (DFS/BFS on matrix)** — count passable regions on a factory
   floor grid with obstacles


## Step 1 — Build weighted factory graph

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_LOCS      = random.randint(10, 16)
N_CHARGERS  = random.randint(2, 4)
GRID_ROWS   = random.randint(6, 9)
GRID_COLS   = random.randint(7, 11)
OBS_DENSITY = round(random.uniform(0.20, 0.35), 2)

LOC_NAMES = [f"L{i+1:02d}" for i in range(N_LOCS)]

# Build random weighted directed graph (connected)
wt_edges = []
order    = list(range(N_LOCS))
random.shuffle(order)
# spanning tree (directed, both ways for connectivity)
for i in range(1, N_LOCS):
    u, v = order[i-1], order[i]
    w1   = random.randint(5, 40)
    w2   = random.randint(5, 40)
    wt_edges.append((u, v, w1))
    wt_edges.append((v, u, w2))

# extra directed edges
for _ in range(N_LOCS):
    u, v = random.sample(range(N_LOCS), 2)
    w    = random.randint(5, 40)
    wt_edges.append((u, v, w))

# Build adjacency list (keep minimum weight for duplicate edges)
wt_adj = defaultdict(dict)
for u, v, w in wt_edges:
    if v not in wt_adj[u] or wt_adj[u][v] > w:
        wt_adj[u][v] = w

# Charger locations
charger_nodes = random.sample(range(N_LOCS), N_CHARGERS)

print(f"Weighted factory graph:")
print(f"  Locations       : {N_LOCS}")
print(f"  Directed edges  : {sum(len(v) for v in wt_adj.values())}")
print(f"  Charger nodes   : {[LOC_NAMES[c] for c in charger_nodes]}")
print()
print(f"  Adjacency list (weight = travel time in seconds):")
for u in range(N_LOCS):
    nbrs = {LOC_NAMES[v]: w for v, w in sorted(wt_adj[u].items())}
    print(f"    {LOC_NAMES[u]}: {nbrs}")


## Step 2 — Dijkstra's shortest path

In [ ]:
def dijkstra(wt_adj, src, n):
    """
    Dijkstra's algorithm using a min-heap.

    dist[v]  = shortest known travel time from src to v.
    heap     = (dist, node) — always expand nearest unvisited node.
    prev[v]  = previous node on shortest path to v.

    O((V + E) log V) time.
    """
    dist = [float('inf')] * n
    prev = [-1] * n
    dist[src] = 0
    heap = [(0, src)]

    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue   # stale entry
        for v, w in wt_adj[u].items():
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(heap, (nd, v))
    return dist, prev

def reconstruct(prev, src, dst):
    path, cur = [], dst
    while cur != -1:
        path.append(cur)
        cur = prev[cur]
    path.reverse()
    return path if path[0] == src else []

# Pick robot start and several destinations
robot_start = random.randint(0, N_LOCS-1)
test_dests  = random.sample([i for i in range(N_LOCS) if i != robot_start],
                             min(5, N_LOCS-1))

dist_from_start, prev_map = dijkstra(wt_adj, robot_start, N_LOCS)

print(f"Dijkstra from {LOC_NAMES[robot_start]}:")
print(f"  {'Destination':<10}  {'Time (s)':>10}  Path")
print(f"  {'─'*10}  {'─'*10}  {'─'*40}")
dijkstra_results = {}
for dst in test_dests:
    d    = dist_from_start[dst]
    path = reconstruct(prev_map, robot_start, dst)
    path_str = " → ".join(LOC_NAMES[p] for p in path) if path else "unreachable"
    dijkstra_results[dst] = (d, path)
    d_str = f"{d}s" if d != float('inf') else "∞"
    print(f"  {LOC_NAMES[dst]:<10}  {d_str:>10}  {path_str}")

reachable = [v for v in range(N_LOCS) if dist_from_start[v] != float('inf')]
avg_time  = round(sum(dist_from_start[v] for v in reachable) / len(reachable), 1)
print(f"\n  Reachable from {LOC_NAMES[robot_start]}: {len(reachable)} / {N_LOCS}")
print(f"  Average shortest time  : {avg_time}s")


## Step 3 — Multi-source BFS: nearest charger for every node

In [ ]:
def multi_source_bfs(net_unweighted, sources, n):
    """
    Multi-source BFS — starts simultaneously from ALL source nodes.

    Initialize queue with all sources at distance 0.
    BFS naturally propagates outward — the first time any node v is reached
    gives the shortest (fewest hop) distance to v from the nearest source.

    O(V + E) time.
    """
    dist   = [-1] * n
    source = [-1] * n   # which charger is nearest
    queue  = deque()
    for s in sources:
        dist[s]   = 0
        source[s] = s
        queue.append(s)

    while queue:
        u = queue.popleft()
        for v in net_unweighted[u]:
            if dist[v] == -1:
                dist[v]   = dist[u] + 1
                source[v] = source[u]
                queue.append(v)
    return dist, source

# Build unweighted undirected version for BFS
unw_adj = defaultdict(list)
for u in wt_adj:
    for v in wt_adj[u]:
        unw_adj[u].append(v)
        unw_adj[v].append(u)

charger_dist, nearest_charger = multi_source_bfs(unw_adj, charger_nodes, N_LOCS)

print(f"Multi-source BFS — nearest charger for each location:")
print(f"  Charger nodes: {[LOC_NAMES[c] for c in charger_nodes]}")
print()
print(f"  {'Location':<10}  {'Nearest charger':>16}  {'Hops':>6}")
print(f"  {'─'*10}  {'─'*16}  {'─'*6}")
for i in range(N_LOCS):
    nc  = nearest_charger[i]
    d   = charger_dist[i]
    nc_str = LOC_NAMES[nc] if nc != -1 else "unreachable"
    d_str  = str(d) if d != -1 else "∞"
    marker = " ★" if i in charger_nodes else ""
    print(f"  {LOC_NAMES[i]:<10}  {nc_str:>16}  {d_str:>6}{marker}")

max_dist_to_charger = max(d for d in charger_dist if d != -1)
farthest_from_charger = LOC_NAMES[[i for i,d in enumerate(charger_dist) if d == max_dist_to_charger][0]]
print(f"\n  Farthest node from any charger: {farthest_from_charger} ({max_dist_to_charger} hops)")


## Step 4 — Grid flood fill: passable regions on factory floor

In [ ]:
def generate_grid(rows, cols, obs_density):
    """Generate factory floor grid: 0=passable, 1=obstacle."""
    grid = []
    for r in range(rows):
        row = []
        for c in range(cols):
            row.append(1 if random.random() < obs_density else 0)
        grid.append(row)
    # ensure at least one passable cell in each corner area
    grid[0][0] = 0
    grid[rows-1][cols-1] = 0
    return grid

def flood_fill_count(grid, rows, cols):
    """
    Count distinct passable regions using BFS flood fill.
    Each call fills one region from an unvisited passable cell.
    Returns (num_regions, region_sizes).
    """
    visited = [[False]*cols for _ in range(rows)]
    dirs    = [(0,1),(0,-1),(1,0),(-1,0)]
    regions = []

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 0 and not visited[r][c]:
                # BFS flood fill
                size  = 0
                queue = deque([(r, c)])
                visited[r][c] = True
                while queue:
                    cr, cc = queue.popleft()
                    size  += 1
                    for dr, dc in dirs:
                        nr, nc = cr+dr, cc+dc
                        if 0 <= nr < rows and 0 <= nc < cols:
                            if grid[nr][nc] == 0 and not visited[nr][nc]:
                                visited[nr][nc] = True
                                queue.append((nr, nc))
                regions.append(size)

    return len(regions), regions

grid = generate_grid(GRID_ROWS, GRID_COLS, OBS_DENSITY)
n_regions, region_sizes = flood_fill_count(grid, GRID_ROWS, GRID_COLS)

# Pretty-print grid
print(f"Factory floor grid ({GRID_ROWS}×{GRID_COLS},  0=passable  ■=obstacle):")
for row in grid:
    print("  " + "".join("■" if c == 1 else "·" for c in row))

print()
print(f"Flood fill — passable regions:")
print(f"  Grid size        : {GRID_ROWS} × {GRID_COLS} = {GRID_ROWS*GRID_COLS} cells")
obstacle_count = sum(grid[r][c] for r in range(GRID_ROWS) for c in range(GRID_COLS))
passable_count = GRID_ROWS*GRID_COLS - obstacle_count
print(f"  Obstacles        : {obstacle_count}  ({obstacle_count/(GRID_ROWS*GRID_COLS)*100:.1f}%)")
print(f"  Passable cells   : {passable_count}")
print(f"  Passable regions : {n_regions}")
print(f"  Region sizes     : {sorted(region_sizes, reverse=True)}")
if region_sizes:
    print(f"  Largest region   : {max(region_sizes)} cells")
    print(f"  Robot can access : {max(region_sizes)/passable_count*100:.1f}% of passable floor from largest region")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " WEIGHTED ROBOT PATH PLANNER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  GRAPH STRUCTURE" + " "*(W-17) + "║")
print(f"║  {'Locations (vertices)':<28}: {N_LOCS:<26}║")
n_edges_total = sum(len(v) for v in wt_adj.values())
print(f"║  {'Directed edges':<28}: {n_edges_total:<26}║")
print(f"║  {'Charger nodes':<28}: {[LOC_NAMES[c] for c in charger_nodes]}{'':<6}║"[:W+2]+"║")
print("╠" + "═"*W + "╣")
print("║  DIJKSTRA RESULTS" + " "*(W-18) + "║")
print(f"║  {'Robot start':<28}: {LOC_NAMES[robot_start]:<26}║")
print(f"║  {'Reachable nodes':<28}: {len(reachable)} / {N_LOCS}{'':<20}║")
print(f"║  {'Avg shortest time':<28}: {avg_time}s{'':<23}║")
best_dest = min(test_dests, key=lambda d: dijkstra_results[d][0])
bd = dijkstra_results[best_dest][0]
print(f"║  {'Nearest checked dest':<28}: {LOC_NAMES[best_dest]} ({bd}s){'':<15}║")
print("╠" + "═"*W + "╣")
print("║  MULTI-SOURCE BFS (CHARGERS)" + " "*(W-29) + "║")
print(f"║  {'Charger count':<28}: {N_CHARGERS:<26}║")
print(f"║  {'Max hops to charger':<28}: {max_dist_to_charger:<26}║")
print(f"║  {'Farthest from charger':<28}: {farthest_from_charger:<26}║")
print("╠" + "═"*W + "╣")
print("║  GRID FLOOD FILL" + " "*(W-17) + "║")
print(f"║  {'Grid size':<28}: {GRID_ROWS}×{GRID_COLS}{'':<22}║")
print(f"║  {'Passable regions':<28}: {n_regions:<26}║")
print(f"║  {'Largest region':<28}: {max(region_sizes) if region_sizes else 0} cells{'':<18}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the network or system?

*Your answer here:*

---

**Q2 — Which graph concept did you find most important, and why?**
Pick one algorithm (BFS, DFS, topological sort, Dijkstra, cycle detection…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
